# Fase 2 - Modelos ML: TF-IDF y GloVe
Implementación y evaluación de LR, RF y SVM con dos representaciones textuales:
TF-IDF de palabras y GloVe SBWC en español. Se evalúa por variante dialectal
(mx, es, cu) para cada tipo de preprocesamiento (normal, stem, lemma).
GridSearchCV con Stratified K-Fold (5 folds) optimizando F1-Macro
(Bhattacharjee et al., 2023; Ortega-Bueno et al., 2022).

El notebook itera automáticamente sobre los tres tipos de preprocesamiento.
GloVe se ejecuta para todos los tipos; se reporta la tasa OOV como diagnóstico.

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import gzip, shutil, os, time
import joblib

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

# gensim no se importa: se usa loader propio para GloVe

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, accuracy_score,
    precision_score, recall_score, classification_report
)


## 2. Configuración

In [2]:
DATA_DIR  = '../data'
GLOVE_GZ  = '../glove-sbwc.i25.vec.gz'
GLOVE_VEC = '../glove-sbwc.i25.vec'

PREPS = ['normal', 'stem', 'lemma']   # se itera automáticamente sobre los tres

FEATURE_COLS = ['n_exc','n_int','n_may','n_emo','n_ris',
                'n_neg','n_elo','n_com','n_pun']

STOP_WORDS = stopwords.words('spanish')
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Preprocesamientos a ejecutar : {PREPS}')
print(f'Features lingüísticos        : {FEATURE_COLS}')


Preprocesamientos a ejecutar : ['normal', 'stem', 'lemma']
Features lingüísticos        : ['n_exc', 'n_int', 'n_may', 'n_emo', 'n_ris', 'n_neg', 'n_elo', 'n_com', 'n_pun']


## 3. Carga de GloVe
Se carga una sola vez antes del bucle principal. Se reporta OOV como diagnóstico
para cada variante de preprocesamiento, pero los experimentos se ejecutan siempre.

(Pennington et al., 2014; Cardellino, 2016)

In [3]:
# Descomprimir si no existe
if not os.path.exists(GLOVE_VEC):
    print('Descomprimiendo GloVe... (puede tardar ~1 min)')
    with gzip.open(GLOVE_GZ, 'rb') as f_in:
        with open(GLOVE_VEC, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print('Descompresión completada.')
else:
    print('Archivo .vec ya existe.')

class GloVeModel:
    def __init__(self, filepath):
        self.key_to_index = {}
        vectors_list = []
        expected_dim = None
        print('Cargando vectores GloVe...')
        t0 = time.time()
        with open(filepath, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i % 100000 == 0 and i > 0:
                    print(f'  {i:,} vectores...', end='\r')
                parts = line.rstrip().split(' ')
                if len(parts) < 2:
                    continue
                # Saltar cabecera formato Word2Vec ("855380 300")
                if len(parts) == 2:
                    try:
                        int(parts[0]); int(parts[1])
                        print(f'Cabecera detectada: {parts[0]} palabras, {parts[1]} dims')
                        continue
                    except ValueError:
                        pass
                word = parts[0]
                try:
                    vector = np.array(parts[1:], dtype=np.float32)
                except ValueError:
                    continue
                if expected_dim is None:
                    expected_dim = len(vector)
                if len(vector) != expected_dim:
                    continue
                self.key_to_index[word] = len(vectors_list)
                vectors_list.append(vector)
        self.vectors     = np.vstack(vectors_list)
        self.vector_size = self.vectors.shape[1]
        print(f'\nListo en {time.time()-t0:.1f}s | '
              f'{len(self.key_to_index):,} vectores | '
              f'{self.vector_size} dimensiones')

    def __contains__(self, word):
        return word in self.key_to_index

    def __getitem__(self, word):
        return self.vectors[self.key_to_index[word]]

glove = GloVeModel(GLOVE_VEC)


Descomprimiendo GloVe... (puede tardar ~1 min)
Descompresión completada.
Cargando vectores GloVe...
Cabecera detectada: 855380 palabras, 300 dims
  800,000 vectores...
Listo en 19.9s | 855,380 vectores | 300 dimensiones


In [4]:
def vectorizar_glove(textos, modelo, dim=300):
    vectores = []
    oov_total, tokens_total = 0, 0
    for texto in textos:
        palabras = str(texto).split()
        tokens_total += len(palabras)
        vecs = [modelo[w] for w in palabras if w in modelo.key_to_index]
        oov_total += len(palabras) - len(vecs)
        vectores.append(np.mean(vecs, axis=0) if vecs else np.zeros(dim))
    oov_rate = oov_total / max(tokens_total, 1) * 100
    return np.array(vectores), oov_rate


## 4. Definición de modelos y grids de hiperparámetros

In [13]:
def build_tfidf_prep():
    return ColumnTransformer([
        ('tfidf_word', TfidfVectorizer(
            stop_words=STOP_WORDS,
            max_features= 10000  # vocabulario completo: ~8k normal, ~5k stem, ~6k lemma
        ), 'MESSAGE_CLEAN'),
        ('ling', 'passthrough', FEATURE_COLS)
    ])

def build_tfidf_pipelines():
    return {
        'LR':  Pipeline([('prep', build_tfidf_prep()),
                         ('clf',  LogisticRegression(max_iter=10000, random_state=42, class_weight='balanced'))]),
        'RF':  Pipeline([('prep', build_tfidf_prep()),
                         ('clf',  RandomForestClassifier(random_state=42, class_weight='balanced'))]),
        'SVM': Pipeline([('prep', build_tfidf_prep()),
                         ('clf',  LinearSVC(random_state=42, class_weight='balanced', max_iter=500000))])
    }

GRIDS_TFIDF = {
    'LR': {
        'prep__tfidf_word__ngram_range': [(1,1), (1,2)],
        'clf__C':                        [0.1, 1.0, 10],
        'clf__solver':                   ['liblinear']
    },
    'RF':  {
        'prep__tfidf_word__ngram_range': [(1,1), (1,2)],
        'clf__n_estimators':             [100, 200, 300],
        'clf__max_depth':                [None, 10, 20],
        'clf__min_samples_split':        [2, 5]
    },
    'SVM': {
        'prep__tfidf_word__ngram_range': [(1,1), (1,2)],
        'clf__C':                        [0.01, 0.1, 1.0, 10]
    }
}

def build_glove_pipelines():
    return {
        'LR':  Pipeline([('scaler', StandardScaler()),
                         ('clf',    LogisticRegression(max_iter=10000, random_state=42, class_weight='balanced'))]),
        'RF':  Pipeline([('scaler', StandardScaler()),
                         ('clf',    RandomForestClassifier(random_state=42, class_weight='balanced'))]),
        'SVM': Pipeline([('scaler', StandardScaler()),
                         ('clf',    LinearSVC(random_state=42, class_weight='balanced', max_iter=100000))])
    }

GRIDS_GLOVE = {
    'LR': {
        'clf__C':        [0.1, 1.0, 10],
        'clf__solver':   ['liblinear']
    },
    'RF': {
        'clf__n_estimators':      [100, 200, 300],
        'clf__max_depth':         [None, 10, 20],
        'clf__min_samples_split': [2, 5]
    },
    'SVM': {'clf__C': [0.01, 0.1, 1.0, 10]}
}

print('Pipelines y grids definidos.')


Pipelines y grids definidos.


## 5. Función principal de experimentos

In [14]:
RESULTADOS = []

def run_experimento(prep, dataset_nombre, df_tr, df_te,
                    repr_nombre, pipelines, grids,
                    X_tr, y_tr, X_te, y_te):
    models_dir = f'{DATA_DIR}/modelos/{prep}'
    os.makedirs(models_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'  PREP={prep} | Dataset={dataset_nombre.upper()} | Repr={repr_nombre}')
    print(f'{"="*60}')

    for modelo_nombre in ['LR', 'RF', 'SVM']:
        pipeline = pipelines[modelo_nombre]
        grid     = grids[modelo_nombre]

        n_cand = 1
        for v in grid.values():
            n_cand *= len(v)
        print(f'\n  → {modelo_nombre} | {n_cand} candidatos...', end=' ')
        t0 = time.time()

        search = GridSearchCV(
            pipeline, grid,
            cv=CV, scoring='f1_macro',
            n_jobs=-1, verbose=0
        )
        search.fit(X_tr, y_tr)

        y_pred = search.best_estimator_.predict(X_te)
        acc    = accuracy_score(y_te, y_pred)
        f1m    = f1_score(y_te, y_pred, average='macro')
        p_ir   = precision_score(y_te, y_pred, pos_label=1, zero_division=0)
        r_ir   = recall_score(y_te, y_pred, pos_label=1, zero_division=0)
        f1_ir  = f1_score(y_te, y_pred, pos_label=1, zero_division=0)

        print(f'F1-Macro CV={search.best_score_:.4f} | '
              f'F1-Macro test={f1m:.4f} | '
              f'tiempo={time.time()-t0:.1f}s')
        print(f'     Mejores params: {search.best_params_}')

        RESULTADOS.append({
            'prep':        prep,
            'dataset':     dataset_nombre,
            'repr':        repr_nombre,
            'modelo':      modelo_nombre,
            'f1_macro_cv': round(search.best_score_, 4),
            'accuracy':    round(acc, 4),
            'f1_macro':    round(f1m, 4),
            'p_ironico':   round(p_ir, 4),
            'r_ironico':   round(r_ir, 4),
            'f1_ironico':  round(f1_ir, 4),
            'best_params': search.best_params_,
            'estimator':   search.best_estimator_
        })

        rep_tag = 'tf-idf' if repr_nombre == 'TF-IDF' else 'glove'
        fname = f'{models_dir}/ml_{rep_tag}_{modelo_nombre.lower()}_{dataset_nombre}.pkl'
        joblib.dump(search.best_estimator_, fname)

print('Función de experimentos lista.')


Función de experimentos lista.


## 6. Bucle principal — todos los preprocesamientos

In [15]:
for PREP in PREPS:
    SUFIJO = '' if PREP == 'normal' else f'_{PREP}'

    print(f'\n{"#"*60}')
    print(f'  PREPROCESAMIENTO: {PREP.upper()}')
    print(f'{"#"*60}')

    # Carga de datasets (sin combinado)
    df_train_mx = pd.read_csv(f'{DATA_DIR}/train_clean{SUFIJO}_mx.csv')
    df_test_mx  = pd.read_csv(f'{DATA_DIR}/test_clean{SUFIJO}_mx.csv')
    df_train_es = pd.read_csv(f'{DATA_DIR}/train_clean{SUFIJO}_es.csv')
    df_test_es  = pd.read_csv(f'{DATA_DIR}/test_clean{SUFIJO}_es.csv')
    df_train_cu = pd.read_csv(f'{DATA_DIR}/train_clean{SUFIJO}_cu.csv')
    df_test_cu  = pd.read_csv(f'{DATA_DIR}/test_clean{SUFIJO}_cu.csv')

    DATASETS = {
        'mx': (df_train_mx, df_test_mx),
        'es': (df_train_es, df_test_es),
        'cu': (df_train_cu, df_test_cu),
    }

    print(f'Datasets cargados ({PREP}):')
    for nombre, (tr, te) in DATASETS.items():
        print(f'  {nombre:4} train={len(tr):,} | test={len(te):,}')

    print(f'\nEjemplo MESSAGE_CLEAN ({PREP}):')
    print(df_train_mx['MESSAGE_CLEAN'].iloc[0])

    # ── Experimentos TF-IDF ──────────────────────────────────
    print(f'\n--- TF-IDF ({PREP}) ---')
    for ds_nombre, (df_tr, df_te) in DATASETS.items():
        X_tr = df_tr[['MESSAGE_CLEAN'] + FEATURE_COLS]
        y_tr = df_tr['IS_IRONIC'].values
        X_te = df_te[['MESSAGE_CLEAN'] + FEATURE_COLS]
        y_te = df_te['IS_IRONIC'].values

        run_experimento(
            prep=PREP,
            dataset_nombre=ds_nombre,
            df_tr=df_tr, df_te=df_te,
            repr_nombre='TF-IDF',
            pipelines=build_tfidf_pipelines(),
            grids=GRIDS_TFIDF,
            X_tr=X_tr, y_tr=y_tr,
            X_te=X_te, y_te=y_te
        )

    # ── Experimentos GloVe ───────────────────────────────────
    # Se ejecuta para todos los PREP. Se reporta OOV como diagnóstico.
    # Con stem/lemma el OOV puede ser elevado; los resultados quedan
    # registrados para análisis comparativo.
    print(f'\n--- GloVe ({PREP}) ---')
    for ds_nombre, (df_tr, df_te) in DATASETS.items():
        print(f'\nVectorizando GloVe: {ds_nombre} ({PREP})...')

        glove_tr, oov_tr = vectorizar_glove(df_tr['MESSAGE_CLEAN'], glove)
        glove_te, oov_te = vectorizar_glove(df_te['MESSAGE_CLEAN'], glove)

        print(f'  OOV train={oov_tr:.1f}% | OOV test={oov_te:.1f}%'
              + (' [OOV alto: texto preprocesado vs vocabulario GloVe]'
                 if oov_tr > 50 else ''))

        X_tr = np.hstack([glove_tr, df_tr[FEATURE_COLS].values.astype(float)])
        y_tr = df_tr['IS_IRONIC'].values
        X_te = np.hstack([glove_te, df_te[FEATURE_COLS].values.astype(float)])
        y_te = df_te['IS_IRONIC'].values

        run_experimento(
            prep=PREP,
            dataset_nombre=ds_nombre,
            df_tr=df_tr, df_te=df_te,
            repr_nombre='GloVe',
            pipelines=build_glove_pipelines(),
            grids=GRIDS_GLOVE,
            X_tr=X_tr, y_tr=y_tr,
            X_te=X_te, y_te=y_te
        )



############################################################
  PREPROCESAMIENTO: NORMAL
############################################################
Datasets cargados (normal):
  mx   train=2,399 | test=600
  es   train=2,398 | test=600
  cu   train=2,400 | test=600

Ejemplo MESSAGE_CLEAN (normal):
rica económicamente pero muy pobre en objetividad

--- TF-IDF (normal) ---

  PREP=normal | Dataset=MX | Repr=TF-IDF

  → LR | 6 candidatos... F1-Macro CV=0.6666 | F1-Macro test=0.6924 | tiempo=1.9s
     Mejores params: {'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 2)}

  → RF | 36 candidatos... F1-Macro CV=0.6438 | F1-Macro test=0.6344 | tiempo=11.5s
     Mejores params: {'clf__max_depth': 20, 'clf__min_samples_split': 2, 'clf__n_estimators': 200, 'prep__tfidf_word__ngram_range': (1, 2)}

  → SVM | 8 candidatos... F1-Macro CV=0.6661 | F1-Macro test=0.6909 | tiempo=6.6s
     Mejores params: {'clf__C': 0.1, 'prep__tfidf_word__ngram_range': (1, 2)}

  PREP=n

## 7. Tablas comparativas y exportación

In [16]:
df_res = pd.DataFrame([{
    'Prep':        r['prep'],
    'Dataset':     r['dataset'],
    'Repr':        r['repr'],
    'Modelo':      r['modelo'],
    'F1-Macro CV': r['f1_macro_cv'],
    'Accuracy':    r['accuracy'],
    'F1-Macro':    r['f1_macro'],
    'F1-Irónico':  r['f1_ironico'],
    'P-Irónico':   r['p_ironico'],
    'R-Irónico':   r['r_ironico'],
} for r in RESULTADOS])

# Tabla por cada tipo de preprocesamiento + mejor modelo por variante
for prep in PREPS:
    subset_prep = df_res[df_res['Prep'] == prep]
    print(f'\n{"="*70}')
    print(f'  TABLA COMPARATIVA — PREP={prep.upper()}')
    print(f'{"="*70}')
    print(subset_prep.drop(columns='Prep').to_string(index=False))

    print(f'\n  MEJOR MODELO POR VARIANTE Y REPRESENTACIÓN (PREP={prep.upper()}):')
    print(f'  {"-"*50}')
    for ds in ['mx', 'es', 'cu']:
        for repr_nombre in ['TF-IDF', 'GloVe']:
            sub = subset_prep[
                (subset_prep['Dataset'] == ds) &
                (subset_prep['Repr'] == repr_nombre)
            ]
            if not sub.empty:
                mejor = sub.loc[sub['F1-Macro'].idxmax()]
                print(f'  {ds:4} {repr_nombre:6}  →  '
                      f'{mejor["Modelo"]:4}  '
                      f'F1-Macro={mejor["F1-Macro"]:.4f}  '
                      f'Accuracy={mejor["Accuracy"]:.4f}')

# Exportar CSV consolidado
out_csv = f'{DATA_DIR}/resultados_ml_todos.csv'
df_res.to_csv(out_csv, index=False)
print(f'\nCSV consolidado guardado: {out_csv}')



  TABLA COMPARATIVA — PREP=NORMAL
Dataset   Repr Modelo  F1-Macro CV  Accuracy  F1-Macro  F1-Irónico  P-Irónico  R-Irónico
     mx TF-IDF     LR       0.6666    0.7283    0.6924      0.5873     0.5918     0.5829
     mx TF-IDF     RF       0.6438    0.6767    0.6344      0.5101     0.5127     0.5075
     mx TF-IDF    SVM       0.6661    0.7267    0.6909      0.5859     0.5888     0.5829
     es TF-IDF     LR       0.7132    0.7367    0.7154      0.6376     0.5890     0.6950
     es TF-IDF     RF       0.7105    0.7450    0.7066      0.6005     0.6284     0.5750
     es TF-IDF    SVM       0.7119    0.7417    0.7227      0.6501     0.5926     0.7200
     cu TF-IDF     LR       0.6587    0.7167    0.6648      0.5330     0.5915     0.4850
     cu TF-IDF     RF       0.6659    0.7167    0.6594      0.5198     0.5974     0.4600
     cu TF-IDF    SVM       0.6594    0.7167    0.6616      0.5251     0.5949     0.4700
     mx  GloVe     LR       0.6038    0.7083    0.6784      0.5803     0.55